# Table 1 – Stroke (demand code 54)
Descriptive statistics stratified by `p1_real_emerg` (real priority 1 assigned by the emergency service).

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

sys.path.insert(0, os.path.abspath('..'))
from kbase.config import settings

warnings.filterwarnings('ignore')

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
DATA_PATH = os.path.join(settings.source_tables_path, settings.stroke_table_cleaned_path)

OUTCOME = "p1_real_emerg"

VARS_NUMERIC = ["age", "year"]

VARS_BINARY  = ["sex", "triage"]

GROUPS = {
    "Day of week": [
        "day_week_monday", "day_week_tuesday", "day_week_wednesday", "day_week_thursday",
        "day_week_friday", "day_week_saturday", "day_week_sunday",
    ],
    "Time of day": [
        "time_of_day_early_morning", "time_of_day_morning",
        "time_of_day_afternoon", "time_of_day_night",
    ],
    "Month": [
        "month_january", "month_february", "month_march", "month_april",
        "month_may", "month_june", "month_july", "month_august",
        "month_september", "month_october", "month_november", "month_december",
    ],
    "Season": [
        "season_spring", "season_summer", "season_autumn", "season_winter",
    ],
    "Patient location": [
        "location_patient_home", "location_patient_public_road", "location_patient_other",
    ],
    "Entity receiving alert": [
        "alert_receiver_112", "alert_receiver_user", "alert_receiver_pol_fg",
        "alert_receiver_hs", "alert_receiver_tele", "alert_receiver_others",
    ],
    "Province": [
        "province_almeria", "province_cadiz", "province_cordoba", "province_granada",
        "province_huelva", "province_jaen", "province_malaga", "province_sevilla",
    ],
}

LABELS_MAP = {
    # Numeric
    "age":                          "Age (years)",
    "year":                         "Year of the call",
    # Binary
    "sex":                          "Sex (1 = male)",
    "triage":                       "Triage performed (1 = yes)",
    # Day of week
    "day_week_monday":              "Monday",
    "day_week_tuesday":             "Tuesday",
    "day_week_wednesday":           "Wednesday",
    "day_week_thursday":            "Thursday",
    "day_week_friday":              "Friday",
    "day_week_saturday":            "Saturday",
    "day_week_sunday":              "Sunday",
    # Time of day
    "time_of_day_early_morning":    "Early morning (00–05 h)",
    "time_of_day_morning":          "Morning (06–11 h)",
    "time_of_day_afternoon":        "Afternoon (12–17 h)",
    "time_of_day_night":            "Night (18–23 h)",
    # Month
    "month_january":                "January",
    "month_february":               "February",
    "month_march":                  "March",
    "month_april":                  "April",
    "month_may":                    "May",
    "month_june":                   "June",
    "month_july":                   "July",
    "month_august":                 "August",
    "month_september":              "September",
    "month_october":                "October",
    "month_november":               "November",
    "month_december":               "December",
    # Season
    "season_spring":                "Spring",
    "season_summer":                "Summer",
    "season_autumn":                "Autumn",
    "season_winter":                "Winter",
    # Patient location
    "location_patient_home":        "Home",
    "location_patient_public_road": "Public road",
    "location_patient_other":       "Other location",
    # Alert receiver
    "alert_receiver_112":           "112",
    "alert_receiver_user":          "User / Patient",
    "alert_receiver_pol_fg":        "Police / Civil Guard",
    "alert_receiver_hs":            "Health services",
    "alert_receiver_tele":          "Tele-assistance",
    "alert_receiver_others":        "Others",
    # Province
    "province_almeria":             "Almería",
    "province_cadiz":               "Cádiz",
    "province_cordoba":             "Córdoba",
    "province_granada":             "Granada",
    "province_huelva":              "Huelva",
    "province_jaen":                "Jaén",
    "province_malaga":              "Málaga",
    "province_sevilla":             "Sevilla",
}

TITLE = "Table 1. Descriptive statistics of stroke calls stratified by real priority (p1_real_emerg)"

OUTPUT_DIR = os.path.join("figures", "04_stroke")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _format_median_iqr(series: pd.Series) -> str:
    s = series.dropna().astype(float)
    if s.empty:
        return ""
    return f"{np.median(s):.1f} [{np.percentile(s, 25):.1f}, {np.percentile(s, 75):.1f}]"


def _fmt_pct(n, d) -> str:
    if d == 0:
        return "0 (0.0%)"
    return f"{int(n)} ({100.0 * n / d:.1f}%)"


def _smd_continuous(x0: pd.Series, x1: pd.Series) -> str:
    x0, x1 = x0.dropna().astype(float), x1.dropna().astype(float)
    pooled_sd = np.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
    if pooled_sd == 0:
        return "NA"
    smd = abs(x1.mean() - x0.mean()) / pooled_sd
    se  = np.sqrt((len(x0) + len(x1)) / (len(x0) * len(x1)) + smd**2 / (2 * (len(x0) + len(x1))))
    return f"{smd:.2f} [{smd - 1.96*se:.2f}, {smd + 1.96*se:.2f}]"


def _smd_binary(p0: float, p1: float, n0: int, n1: int) -> str:
    denom = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)) / 2)
    if denom == 0:
        return "NA"
    smd = abs(p1 - p0) / denom
    se  = np.sqrt(1 / (2 * n0) + 1 / (2 * n1))
    return f"{smd:.2f} [{smd - 1.96*se:.2f}, {smd + 1.96*se:.2f}]"


def _pvalue_str(p: float) -> str:
    return "<0.001" if p < 0.001 else f"{p:.3f}"


In [ ]:
# ---------------------------------------------------------------------------
# Main function
# ---------------------------------------------------------------------------

def build_table1(
    data_path:    str,
    outcome:      str,
    vars_numeric: list,
    vars_binary:  list,
    groups:       dict,
    labels_map:   dict,
    title:        str,
    output_dir:   str,
) -> pd.DataFrame:
    """
    Build Table 1 stratified by a binary outcome.

    Numeric vars  → median [Q1, Q3] | Mann-Whitney U | SMD continuous
    Binary vars   → N (%)           | Chi-square / Fisher | SMD proportions
    Grouped vars  → group header row + one N(%) row per dummy
    """

    # --- Load & filter ---
    df = pq.read_table(data_path).to_pandas()
    if outcome not in df.columns:
        raise ValueError(f"Outcome column '{outcome}' not found.")
    df = df.loc[df[outcome].isin([0, 1])].copy()

    g0, g1   = df[df[outcome] == 0], df[df[outcome] == 1]
    n_all, n0, n1 = len(df), len(g0), len(g1)

    col_overall = f"Overall (N={n_all:,})"
    col_g0      = f"{outcome}=0 (N={n0:,})"
    col_g1      = f"{outcome}=1 (N={n1:,})"
    col_smd     = "SMD [95% CI]"
    col_p       = "p-value"

    all_vars_in_order = (
        vars_numeric
        + vars_binary
        + [v for vlist in groups.values() for v in vlist]
    )

    # Check which vars exist
    missing = [v for v in all_vars_in_order if v not in df.columns]
    if missing:
        print(f"[WARNING] Variables not found and skipped: {missing}")

    rows       = []   # data rows
    row_types  = []   # "header" | "numeric" | "binary"

    def _add_row(label, overall, c0, c1, smd, p, rtype):
        rows.append({
            "Variable": label,
            col_overall: overall,
            col_g0:      c0,
            col_g1:      c1,
            col_smd:     smd,
            col_p:       p,
        })
        row_types.append(rtype)

    # ── Numeric variables ────────────────────────────────────────────────────
    for v in vars_numeric:
        if v not in df.columns:
            continue
        x0, x1 = g0[v], g1[v]
        _, pval = stats.mannwhitneyu(
            x0.dropna().astype(float), x1.dropna().astype(float), alternative="two-sided"
        )
        _add_row(
            label   = labels_map.get(v, v),
            overall = _format_median_iqr(df[v]),
            c0      = _format_median_iqr(x0),
            c1      = _format_median_iqr(x1),
            smd     = _smd_continuous(x0, x1),
            p       = _pvalue_str(pval),
            rtype   = "numeric",
        )

    # ── Standalone binary variables ──────────────────────────────────────────
    for v in vars_binary:
        if v not in df.columns:
            continue
        n_all_1 = int((df[v] == 1).sum())
        n0_1    = int((g0[v] == 1).sum())
        n1_1    = int((g1[v] == 1).sum())
        p0, p1  = n0_1 / n0, n1_1 / n1

        tbl = np.array([[n1_1, n1 - n1_1], [n0_1, n0 - n0_1]])
        if tbl.min() < 5:
            _, pval = stats.fisher_exact(tbl)
        else:
            _, pval, _, _ = stats.chi2_contingency(tbl)

        _add_row(
            label   = labels_map.get(v, v),
            overall = _fmt_pct(n_all_1, n_all),
            c0      = _fmt_pct(n0_1, n0),
            c1      = _fmt_pct(n1_1, n1),
            smd     = _smd_binary(p0, p1, n0, n1),
            p       = _pvalue_str(pval),
            rtype   = "binary",
        )

    # ── Grouped one-hot variables ────────────────────────────────────────────
    for group_label, members in groups.items():
        present = [v for v in members if v in df.columns]
        if not present:
            continue

        # Group header (no stats, just a label row)
        _add_row(group_label, "", "", "", "", "", rtype="header")

        for v in present:
            n_all_1 = int((df[v] == 1).sum())
            n0_1    = int((g0[v] == 1).sum())
            n1_1    = int((g1[v] == 1).sum())
            p0, p1  = n0_1 / n0, n1_1 / n1

            tbl = np.array([[n1_1, n1 - n1_1], [n0_1, n0 - n0_1]])
            if tbl.min() < 5:
                _, pval = stats.fisher_exact(tbl)
            else:
                _, pval, _, _ = stats.chi2_contingency(tbl)

            _add_row(
                label   = "  " + labels_map.get(v, v),
                overall = _fmt_pct(n_all_1, n_all),
                c0      = _fmt_pct(n0_1, n0),
                c1      = _fmt_pct(n1_1, n1),
                smd     = _smd_binary(p0, p1, n0, n1),
                p       = _pvalue_str(pval),
                rtype   = "binary",
            )

    result = pd.DataFrame(rows)

    # ── 1. Console print ─────────────────────────────────────────────────────
    print(title)
    print("=" * len(title))
    print(result.to_string(index=False))
    print("\nNumeric: median [Q1, Q3] | Mann-Whitney U test.")
    print("Binary / grouped: N (%) | Chi-square or Fisher's exact test.")
    print("SMD = standardised mean difference [95% CI].")

    # ── 2. Word export ───────────────────────────────────────────────────────
    doc  = Document()
    style = doc.styles['Normal']
    style.font.name = 'Calibri'
    style.font.size = Pt(10)

    tp = doc.add_paragraph()
    tr = tp.add_run(title)
    tr.bold = True
    tr.font.size = Pt(11)
    doc.add_paragraph()

    n_cols = len(result.columns)
    table  = doc.add_table(rows=len(result) + 1, cols=n_cols)
    table.style = 'Table Grid'

    HDR_COLOR  = RGBColor(0x4F, 0x81, 0xBD)
    GRP_COLOR  = RGBColor(0x4F, 0x81, 0xBD)
    ROW_COLOR  = RGBColor(0xDC, 0xE6, 0xF1)

    def _shd(cell, rgb):
        tc   = cell._tc
        tcPr = tc.get_or_add_tcPr()
        shd  = OxmlElement('w:shd')
        shd.set(qn('w:val'),   'clear')
        shd.set(qn('w:color'), 'auto')
        shd.set(qn('w:fill'),  f"{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}")
        tcPr.append(shd)

    # Header row
    for i, col_name in enumerate(result.columns):
        cell = table.rows[0].cells[i]
        cell.text = col_name
        run = cell.paragraphs[0].runs[0]
        run.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.size = Pt(9)
        _shd(cell, HDR_COLOR)

    # Data rows
    for ri, (_, row_data) in enumerate(result.iterrows()):
        rtype = row_types[ri]
        cells = table.rows[ri + 1].cells
        for ci, value in enumerate(row_data):
            cells[ci].text = str(value)
            run = cells[ci].paragraphs[0].runs[0]
            run.font.size = Pt(9)
            if rtype == "header":
                run.bold = True
                run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
                _shd(cells[ci], GRP_COLOR)
            else:
                _shd(cells[ci], ROW_COLOR)

    doc.add_paragraph()
    note = doc.add_paragraph()
    nr = note.add_run(
        "Numeric variables: median [Q1, Q3]; Mann–Whitney U test. "
        "Binary / grouped variables: N (%); Chi-square or Fisher's exact test. "
        "SMD = standardised mean difference [95% CI]."
    )
    nr.italic = True
    nr.font.size = Pt(9)

    word_path = os.path.join(output_dir, "table1_stroke.docx")
    doc.save(word_path)
    print(f"\n[Saved] Word → {word_path}")

    # ── 3. JPG export ────────────────────────────────────────────────────────
    fig_h = max(3, len(result) * 0.45 + 1.5)
    fig, ax = plt.subplots(figsize=(17, fig_h))
    ax.axis("off")

    mpl_table = ax.table(
        cellText  = result.values,
        colLabels = result.columns,
        cellLoc   = "center",
        loc       = "center",
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8)
    mpl_table.auto_set_column_width(col=list(range(len(result.columns))))

    for (r, c), cell in mpl_table.get_celld().items():
        cell.set_edgecolor("#AAAAAA")
        if r == 0:
            cell.set_facecolor("#4F81BD")
            cell.set_text_props(weight="bold", color="white")
        elif row_types[r - 1] == "header":
            cell.set_facecolor("#4F81BD")
            cell.set_text_props(weight="bold", color="white")
        else:
            cell.set_facecolor("#DCE6F1")
            cell.set_text_props(color="black")

    fig.suptitle(title, fontsize=9, weight="bold", y=0.99)
    plt.tight_layout()

    jpg_path = os.path.join(output_dir, "table1_stroke.jpg")
    fig.savefig(jpg_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[Saved] JPG  → {jpg_path}")

    return result


In [ ]:
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
table1 = build_table1(
    data_path    = DATA_PATH,
    outcome      = OUTCOME,
    vars_numeric = VARS_NUMERIC,
    vars_binary  = VARS_BINARY,
    groups       = GROUPS,
    labels_map   = LABELS_MAP,
    title        = TITLE,
    output_dir   = OUTPUT_DIR,
)
